OR calculation across the CVFG dataframe

In [1]:
# Make sure our package is installed (run once)
#!pip install -e /home/dataproc/repos/igvf-cvfg-biobank

Notes on running in AoU's DataProc: as of 2026-09-25, trying to use pandas with the `fastparquet` engine throws an error. Manually forcing an update of relevant packages resolves it:

In [2]:
# Run once in dataproc
#!pip install --upgrade numpy numba fastparquet pandas statsmodels

In the standard environment, `fastparquet` is not installed, but we use it to append to a parquet file:

In [3]:
# Run once in a standard env
#!pip install fastparquet

In [4]:
import os
import pandas as pd
from pathlib import Path
from cvfgaou import notation

In [5]:
# In standard env
#wgs_mt_path = "/home/jupyter/workspace/All of Us Controlled Tier v8/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt"

In [6]:
# In Dataproc
wgs_mt_path = 'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt'

In [7]:
# Constants (mostly paths to things)

variant_table_path = 'variant_tables/integrated_variant_effect_biobank_input_data.20260917.tsv.gz'
google_project = os.getenv('GOOGLE_PROJECT')
functional_carriers_path = Path('carrier_tables/functional-carriers_2026-10-08.parquet')
variant_counts_path = 'variant_groups/functional-variant-counts_2026-10-08.parquet'

In [8]:
# Load dataframe
variants_df = pd.read_table(variant_table_path, dtype=str)

In [9]:
variants_df

Now establish the sets of variants to test.

First we inspect the columns that will define variant sets:

In [10]:
variants_df[['condensed_consequence']].value_counts(dropna=False)

In [11]:
variants_df[['auth_reported_func_class_category']].value_counts(dropna=False)

In [12]:
variants_df['ExCALIBR_points'].value_counts(dropna=False)

In [13]:
# Limit to genes with case/control cohorts established
genes_with_cohorts = {
    gene for gene in variants_df['Gene'].unique()
    if Path('gene_cohorts', f'{gene} cohort.parquet').exists()
}

In [14]:
genes_with_cohorts

For each dataset, we examine the following variant sets are by functional category:

- Normal function (all)
- Abnormal function:
  - All
  - Missense
  - Splicing
  - Stop gain
  - Stop gain / deletion / splicing

In [15]:
variant_groups = {
    (dataset, gene, 'Functional class', consequence, classification):
        df[classification_selection]
    for (dataset, gene), df in variants_df.groupby(['Dataset', 'Gene'])
    for classification, consequence, classification_selection in (
        ('Functionally Normal', 'All', df['auth_reported_func_class_category'] == 'normal'),
        ('Functionally Abnormal', 'All', df['auth_reported_func_class_category'] == 'abnormal'),
        ('Functionally Abnormal', 'Missense',
         (df['auth_reported_func_class_category'] == 'abnormal') &
         (df['condensed_consequence'] == 'missense_variant')
        ),
        ('Functionally Abnormal', 'Truncating',
         (df['auth_reported_func_class_category'] == 'abnormal') &
         (df['condensed_consequence'] == 'stop_gained')
        ),
        ('Functionally Abnormal', 'Inframe indel',
         (df['auth_reported_func_class_category'] == 'abnormal') &
         (df['condensed_consequence'] == 'inframe_indel')
        ),
        ('Functionally Abnormal', 'Splicing',
         (df['auth_reported_func_class_category'] == 'abnormal') &
         df['condensed_consequence'].isin(('splicing_variant', 'splice_site_variant'))
        ),
        ('Functionally Abnormal', 'Non-truncating',
         (df['auth_reported_func_class_category'] == 'abnormal') &
         ~(df['condensed_consequence'] == 'stop_gained')
        ),
        ('Functionally Abnormal', 'Non-missense',
         (df['auth_reported_func_class_category'] == 'abnormal') &
         ~(df['condensed_consequence'] == 'missense_variant')
        )
    )
    if gene in genes_with_cohorts
}

Next, get the sets of carriers for each variant group

In [16]:
# First, make sure the position column is an integer and
# make sure that the chromosomes are strings with the "chr" prefix

variant_groups = {
    group_label: df.assign(
        Chromosome = 'chr' + df['Chrom'],
        pos = pd.to_numeric(df['hg38_start'], downcast='integer')
    )
    for group_label, df in variant_groups.items()
}

In [17]:
# Check for missing NT coordinates
for label, df in variant_groups.items():
    if df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']].isna().any(axis=None):
        display(label)
        display(df)

In [18]:
# Save variant sets for each group
variant_group_counts = {
    group_label: df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']].drop_duplicates().shape[0]
    for group_label, df in variant_groups.items()
}

In [19]:
pd.Series(variant_group_counts).to_frame(name="Variants in group").to_parquet(variant_counts_path)

In [20]:
# Drop empty groups
variant_groups = {
    group_label: df
    for group_label, df in variant_groups.items()
    if not df.empty
}

In [21]:
import os
import hail as hl
from tqdm.notebook import tqdm

In [22]:
hl.init(
    #default_reference='GRCh38', deprecated
    gcs_requester_pays_configuration=google_project
)

In [23]:
# WGS table

wgs_mt = hl.read_matrix_table(wgs_mt_path)

In [24]:
from cvfgaou import hailtools

In [25]:
# Index column labels
index_cols=['Dataset/Predictor', 'Gene', 'Grouping/Calibration', 'Consequence', 'Group/Range']

# Attempt to check if we are adding to an existing file
excluded_groups = set()
if functional_carriers_path.exists():
    carriers_df = pd.read_parquet(functional_carriers_path)
    excluded_groups = set(
        carriers_df.
        index.
        to_frame()[index_cols].
        drop_duplicates().
        itertuples(index=False, name=None)
    )
    display(f'Existing file found with {len(excluded_groups)} groups.')

for variant_group, df in tqdm(variant_groups.items()):
    if variant_group in excluded_groups:
        continue
    carrier_df, _ = hailtools.get_detailed_cols_with_variants(
        df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']], wgs_mt,
        contig_col='Chromosome',
        pos_col='pos',
        ref_col='ref_allele',
        alt_col='alt_allele',
        reference_genome='GRCh38'
    ).set_index('s')
    labeled_carrier_df = pd.concat(
        {variant_group: carrier_df},
        names=index_cols
    )

    labeled_carrier_df.to_parquet(
        functional_carriers_path,
        engine='fastparquet',
        append=functional_carriers_path.exists()
    )